# Biomedical KG Data QA Notebook

This notebook is for inspecting source downloads, validating normalized graph CSVs, checking duplicate IDs, and reviewing row counts before Neo4j load.

In [ ]:
from pathlib import Path
import polars as pl

STORAGE_ROOT = Path('/app/db_sample/Neo4j')
LATEST_PATH = STORAGE_ROOT / 'latest_run.json'
latest = pl.read_json(LATEST_PATH) if LATEST_PATH.exists() else None
RUN_ROOT = Path(latest['run_root_dir'][0]) if latest is not None and latest.height else STORAGE_ROOT
RAW_DIR = RUN_ROOT / 'raw'
PROCESSED_DIR = RUN_ROOT / 'processed'
NODES_DIR = PROCESSED_DIR / 'nodes'
EDGES_DIR = PROCESSED_DIR / 'edges'
RELATIONS_DIR = PROCESSED_DIR / 'relations'

In [ ]:
sorted(str(path.relative_to(RAW_DIR)) for path in RAW_DIR.rglob('*') if path.is_file())[:50]

In [ ]:
node_frames = [pl.read_csv(path).with_columns(pl.lit(path.name).alias('file_name')) for path in NODES_DIR.glob('*.csv')]
edge_frames = [pl.read_csv(path).with_columns(pl.lit(path.name).alias('file_name')) for path in EDGES_DIR.glob('*.csv')]
relation_frames = [pl.read_csv(path).with_columns(pl.lit(path.name).alias('file_name')) for path in RELATIONS_DIR.glob('*.csv')]

nodes = pl.concat(node_frames, how='diagonal_relaxed') if node_frames else pl.DataFrame()
edges = pl.concat(edge_frames, how='diagonal_relaxed') if edge_frames else pl.DataFrame()
relations = pl.concat(relation_frames, how='diagonal_relaxed') if relation_frames else pl.DataFrame()

nodes.shape, edges.shape, relations.shape

In [ ]:
nodes.group_by('label').count().sort('count', descending=True)

In [ ]:
edges.group_by('relationship_type').count().sort('count', descending=True)

In [ ]:
duplicate_nodes = nodes.group_by('node_id').count().filter(pl.col('count') > 1)
duplicate_edges = edges.group_by('edge_id').count().filter(pl.col('count') > 1)
duplicate_relations = relations.group_by('relation_id').count().filter(pl.col('count') > 1)

duplicate_nodes.height, duplicate_edges.height, duplicate_relations.height

In [ ]:
nodes.select(['node_id', 'label', 'name', 'source']).head(20)